# FID Score: Standard GAN Evaluation Metric

Frechet Inception Distance (FID) is the industry-standard metric for evaluating GAN quality.

## What is FID?
- Measures similarity between real and synthetic image distributions
- Uses InceptionV3 features (pre-trained on ImageNet)
- Lower FID = Better GAN quality

## Interpretation:
- FID < 10: Excellent
- FID 10-50: Good  
- FID 50-100: Moderate
- FID > 100: Poor

This is NOT binary classification - it's distribution matching!


In [1]:
import numpy as np
import cv2
from scipy import linalg
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tqdm import tqdm
import glob
import random

print("Libraries loaded!")


2025-10-30 20:33:21.844073: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761870801.860768 3599080 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761870801.865924 3599080 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761870801.877970 3599080 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761870801.877982 3599080 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761870801.877984 3599080 computation_placer.cc:177] computation placer alr

Libraries loaded!


## Load InceptionV3 Model


In [2]:
# Load InceptionV3 for feature extraction
inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))

print("InceptionV3 model loaded")


I0000 00:00:1761870805.043898 3599080 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 871 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:07:00.0, compute capability: 8.0


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
InceptionV3 model loaded


## FID Calculation Functions


In [3]:
def calculate_fid(real_features, synthetic_features):
    """
    Calculate FID score between real and synthetic images.
    
    Args:
        real_features: Features from real images
        synthetic_features: Features from synthetic images
    
    Returns:
        fid: FID score (lower is better)
    """
    # Calculate mean and covariance
    mu_real = np.mean(real_features, axis=0)
    mu_syn = np.mean(synthetic_features, axis=0)
    
    sigma_real = np.cov(real_features, rowvar=False)
    sigma_syn = np.cov(synthetic_features, rowvar=False)
    
    # Calculate FID
    diff = mu_real - mu_syn
    covmean, _ = linalg.sqrtm(sigma_real.dot(sigma_syn), disp=False)
    
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_syn - 2 * covmean)
    
    return fid

def extract_features(image_paths, model, batch_size=32):
    """
    Extract InceptionV3 features from images.
    
    Args:
        image_paths: List of image file paths
        model: InceptionV3 model
        batch_size: Batch size for processing
    
    Returns:
        features: Extracted features
    """
    features = []
    
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting features"):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        
        for img_path in batch_paths:
            # Load grayscale image
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            # Convert to RGB (InceptionV3 expects 3 channels)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            # Resize to 299x299 (InceptionV3 input size)
            img_resized = cv2.resize(img_rgb, (299, 299))
            # Preprocess
            img_preprocessed = preprocess_input(img_resized)
            batch_images.append(img_preprocessed)
        
        batch_images = np.array(batch_images)
        batch_features = model.predict(batch_images, verbose=0)
        features.extend(batch_features)
    
    return np.array(features)

print("FID functions defined")


FID functions defined


## Load Images and Calculate FID


In [4]:
# Paths
SYNTHETIC_PATH = '/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_synthetic_resized/'
ORIGINAL_PATH = '/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/full_original_method3/'

# Load all images
syn_files = sorted(glob.glob(SYNTHETIC_PATH + '*.png'))
orig_files = sorted(glob.glob(ORIGINAL_PATH + '*.png'))

# Sample 10,000 from each for FID calculation (more = better estimate)
random.seed(42)
num_samples = min(10000, len(syn_files), len(orig_files))

selected_syn = random.sample(syn_files, num_samples)
selected_orig = random.sample(orig_files, num_samples)

print(f"Selected {num_samples} images from each dataset")
print("Calculating FID score (this may take 10-20 minutes)...")

# Extract features
print("\\nExtracting features from synthetic images...")
syn_features = extract_features(selected_syn, inception_model)

print("\\nExtracting features from original images...")
orig_features = extract_features(selected_orig, inception_model)

# Calculate FID
fid_score = calculate_fid(orig_features, syn_features)

print("\\n" + "="*70)
print("FID SCORE RESULTS")
print("="*70)
print(f"FID Score: {fid_score:.4f}")
print("\\nInterpretation:")
if fid_score < 10:
    print("  Excellent GAN quality")
elif fid_score < 50:
    print("  Good GAN quality")
elif fid_score < 100:
    print("  Moderate GAN quality")
else:
    print("  Poor GAN quality - significant distribution differences")
print("="*70)

# Save result
with open('/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic/classifier_results/fid_score.txt', 'w') as f:
    f.write(f"FID Score: {fid_score:.4f}\\n")
    f.write(f"Samples used: {num_samples} from each class\\n")

print(f"\\nFID score saved")


Selected 10000 images from each dataset
Calculating FID score (this may take 10-20 minutes)...
\nExtracting features from synthetic images...


Extracting features:   0%|                                                                                                                              | 0/313 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1761870810.749324 3599130 service.cc:152] XLA service 0x7f6c5c00a480 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761870810.749359 3599130 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2025-10-30 20:33:30.848392: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1761870811.645245 3599130 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-10-30 20:33:32.095115: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 493.36MiB with freed_by_

\nExtracting features from original images...


Extracting features: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 313/313 [01:10<00:00,  4.46it/s]


\n======================================================================
FID SCORE RESULTS
FID Score: 105.1671
\nInterpretation:
  Poor GAN quality - significant distribution differences
\nFID score saved
